In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("Data/DateFruit_Dataset.csv")

In [4]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [5]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [6]:
X = df.drop(columns=["Class"], axis=1)
y=df["Class"]

In [27]:
X.shape

(898, 34)

In [7]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [8]:
le = LabelEncoder()

y = le.fit_transform(y)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
X_train_scaled.shape

(718, 34)

# PCA

In [56]:
from sklearn.decomposition import PCA

pca = PCA()

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# ANN

In [57]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [58]:
X_train_tensor = torch.tensor(X_train_pca, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_pca, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [59]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [60]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [61]:
# build our model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train_pca.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
            
        )

    def forward(self, x):
        return self.model(x)     

In [62]:
model = ANN()

#loss & optim

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [63]:
epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss /len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, {train_loss}")

epoch = 1/100, 1.7616773273633874
epoch = 2/100, 1.286563938078673
epoch = 3/100, 0.9076246308243793
epoch = 4/100, 0.6577793217223623
epoch = 5/100, 0.49811738340750983
epoch = 6/100, 0.4021442338176396
epoch = 7/100, 0.3400428178517715
epoch = 8/100, 0.2993965187798376
epoch = 9/100, 0.26782817166784534
epoch = 10/100, 0.23888674054456793
epoch = 11/100, 0.216282375804756
epoch = 12/100, 0.20830110136581503
epoch = 13/100, 0.19018317726643189
epoch = 14/100, 0.17618113853361295
epoch = 15/100, 0.16579557598932929
epoch = 16/100, 0.15848080830081648
epoch = 17/100, 0.1463155743220578
epoch = 18/100, 0.14105420430069385
epoch = 19/100, 0.14634754200992378
epoch = 20/100, 0.12902661493938902
epoch = 21/100, 0.12398042327359966
epoch = 22/100, 0.12592838170087856
epoch = 23/100, 0.11862858482029127
epoch = 24/100, 0.11503266697020634
epoch = 25/100, 0.10896553876607315
epoch = 26/100, 0.10932067117613295
epoch = 27/100, 0.10055953442402508
epoch = 28/100, 0.09780939441660176
epoch = 29/1

In [64]:
#eveluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
      
        total += yb.size(0)

print("accuracy : ", correct/total *100)

accuracy :  94.44444444444444


In [34]:
y_test.shape

(180,)